# Clonogenic Scan Pipeline в Google Colab

Этот ноутбук подготовлен для пользователей без локальной настройки Java/Python.

Что нужно подготовить заранее:
- TIFF-сканы;
- `well_layout.csv` со схемой лунок;
- `plate_manifest.csv` с метаданными по каждой лунке;
- при желании собственный `settings.properties`, но базовые параметры можно задать прямо в форме ноутбука.

Важно:
- поддерживаются только `.tif` / `.tiff`;
- для новых пользователей рекомендуется режим `manifest + layout`;
- CDI и heatmap нужно включать только если эксперимент действительно имеет структуру доза × условие.

## 1. Установка кода и зависимостей

In [ ]:
# @title Клонировать репозиторий и поставить зависимости
REPO_URL = "https://github.com/OWNER/clonogenic_scan_pipeline.git" # @param {type:"string"}
REPO_BRANCH = "main" # @param {type:"string"}

import os
import shutil
import subprocess
import sys
import urllib.request
from pathlib import Path

if "OWNER/clonogenic_scan_pipeline.git" in REPO_URL:
    raise ValueError("Замените REPO_URL на реальный адрес вашего GitHub-репозитория.")

workspace = Path('/content/clonogenic_scan_pipeline')
if workspace.exists():
    shutil.rmtree(workspace)

subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(workspace)], check=True)
subprocess.run(['apt-get', 'update', '-y'], check=True)
subprocess.run(['apt-get', 'install', '-y', 'openjdk-17-jdk-headless'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(workspace / 'requirements.txt')], check=True)

ij_jar = Path('/content/ij-1.54p.jar')
urllib.request.urlretrieve('https://repo1.maven.org/maven2/net/imagej/ij/1.54p/ij-1.54p.jar', ij_jar)

os.environ['FIJI_IJ_JAR'] = str(ij_jar)
os.environ['CLONO_REPO'] = str(workspace)
os.environ['CLONO_INPUT'] = '/content/clonogenic_input'
os.environ['CLONO_META'] = '/content/clonogenic_metadata'
os.environ['CLONO_OUTPUT'] = '/content/clonogenic_output'

Path(os.environ['CLONO_INPUT']).mkdir(parents=True, exist_ok=True)
Path(os.environ['CLONO_META']).mkdir(parents=True, exist_ok=True)
Path(os.environ['CLONO_OUTPUT']).mkdir(parents=True, exist_ok=True)

print('Готово.')
print('Репозиторий:', workspace)
print('ImageJ jar:', ij_jar)


## 2. Шаблоны для заполнения

Если у вас ещё нет `well_layout.csv` и `plate_manifest.csv`, выполните ячейку ниже. Она сохранит шаблоны из папки `examples/` и предложит скачать их.

In [ ]:
import os
import shutil
from pathlib import Path
from google.colab import files

repo = Path(os.environ['CLONO_REPO'])
template_dir = Path('/content/clonogenic_templates')
if template_dir.exists():
    shutil.rmtree(template_dir)
template_dir.mkdir(parents=True, exist_ok=True)

for name in ['settings.properties', 'well_layout.csv', 'plate_manifest.csv']:
    shutil.copy2(repo / 'examples' / name, template_dir / name)

archive = shutil.make_archive('/content/clonogenic_templates', 'zip', root_dir=template_dir)
print('Шаблоны подготовлены:', archive)
files.download(archive)


## 3. Загрузка TIFF-сканов

In [ ]:
import os
import shutil
from pathlib import Path
from google.colab import files

input_dir = Path(os.environ['CLONO_INPUT'])
if input_dir.exists():
    shutil.rmtree(input_dir)
input_dir.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
saved = []
for name, payload in uploaded.items():
    if not name.lower().endswith(('.tif', '.tiff')):
        print('Пропущен не-TIFF файл:', name)
        continue
    target = input_dir / Path(name).name
    target.write_bytes(payload)
    saved.append(target.name)

if not saved:
    raise ValueError('Не загружено ни одного TIFF-файла.')

print('Загружены TIFF-файлы:')
for name in sorted(saved):
    print(' -', name)


## 4. Загрузка `well_layout.csv` и `plate_manifest.csv`

Здесь можно загрузить готовые CSV-файлы, заполненные в Excel или Google Sheets.

In [ ]:
import os
import shutil
from pathlib import Path
from google.colab import files

meta_dir = Path(os.environ['CLONO_META'])
if meta_dir.exists():
    shutil.rmtree(meta_dir)
meta_dir.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
for name, payload in uploaded.items():
    target = meta_dir / Path(name).name
    target.write_bytes(payload)

print('Загруженные метафайлы:')
for path in sorted(meta_dir.glob('*')):
    print(' -', path.name)


## 5. Основные параметры запуска

In [ ]:
# @title Основные пользовательские переменные
DATASET = 'hf' # @param ['hf', 'zr', 'auto']
OUTPUT_FOLDER = 'run_01' # @param {type:"string"}
RUN_CDI = False # @param {type:"boolean"}
MATERIAL = 'Material' # @param {type:"string"}
CELL_LINE = '4T1' # @param {type:"string"}

# Ключевые параметры детекции.
MIN_COLONY_AREA = 14 # @param {type:"integer"}
MIN_AREA_SCALE_SCAN = 0.76 # @param {type:"number"}
MIN_AREA_SCALE_PHOTO = 0.55 # @param {type:"number"}
WELL_CROP_SCALE = 1.28 # @param {type:"number"}
SCAN_THRESHOLD_FACTOR = 0.95 # @param {type:"number"}

# По каким полям из manifest усреднять лунки внутри одной пластины.
SUMMARY_GROUP_FIELDS = 'concentration' # @param {type:"string"}

print('DATASET =', DATASET)
print('RUN_CDI =', RUN_CDI)
print('OUTPUT_FOLDER =', OUTPUT_FOLDER)


## 6. Сформировать runtime-config

Ноутбук сам создаёт `settings.properties` из формы выше. Если нужно, потом можно открыть этот файл и посмотреть, что именно пошло в Java-анализатор.

In [ ]:
import os
from pathlib import Path

meta_dir = Path(os.environ['CLONO_META'])
settings_path = meta_dir / 'runtime_settings.properties'
settings_path.write_text(
    '\n'.join([
        f'dataset_mode={DATASET}',
        f'min_colony_area={MIN_COLONY_AREA}',
        f'min_area_scale_scan={MIN_AREA_SCALE_SCAN}',
        f'min_area_scale_photo={MIN_AREA_SCALE_PHOTO}',
        f'well_crop_scale={WELL_CROP_SCALE}',
        f'scan_threshold_factor={SCAN_THRESHOLD_FACTOR}',
        f'summary_group_fields={SUMMARY_GROUP_FIELDS}',
    ]) + '\n',
    encoding='utf-8'
)

layout_path = meta_dir / 'well_layout.csv'
manifest_path = meta_dir / 'plate_manifest.csv'

if RUN_CDI and 'concentration' not in [part.strip() for part in SUMMARY_GROUP_FIELDS.split(',') if part.strip()]:
    raise ValueError('Для CDI-режима поле concentration должно входить в SUMMARY_GROUP_FIELDS.')

if not layout_path.exists():
    raise FileNotFoundError('Не найден well_layout.csv. Загрузите его на шаге 4.')
if not manifest_path.exists():
    raise FileNotFoundError('Не найден plate_manifest.csv. Загрузите его на шаге 4.')

print('settings.properties:', settings_path)
print('well_layout.csv:', layout_path)
print('plate_manifest.csv:', manifest_path)


## 7. Запуск анализа

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

repo = Path(os.environ['CLONO_REPO'])
input_dir = Path(os.environ['CLONO_INPUT'])
meta_dir = Path(os.environ['CLONO_META'])
output_root = Path(os.environ['CLONO_OUTPUT'])
output_dir = output_root / OUTPUT_FOLDER

if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    'bash',
    str(repo / 'scripts' / 'run_material_pipeline.sh'),
    '--input-dir', str(input_dir),
    '--output-dir', str(output_dir),
    '--dataset', DATASET,
    '--config', str(meta_dir / 'runtime_settings.properties'),
    '--layout', str(meta_dir / 'well_layout.csv'),
    '--manifest', str(meta_dir / 'plate_manifest.csv'),
]

if RUN_CDI:
    cmd += ['--material', MATERIAL, '--cell-line', CELL_LINE]
else:
    cmd += ['--skip-cdi']

print('Запуск:')
print(' '.join(cmd))
subprocess.run(cmd, check=True, cwd=repo)
print('Готово. Результаты лежат в', output_dir)


## 8. Просмотр результатов

In [ ]:
import os
from pathlib import Path
from IPython.display import Image, Markdown, display

output_root = Path(os.environ['CLONO_OUTPUT'])
output_dir = output_root / OUTPUT_FOLDER

heatmap_path = output_dir / f'cdi_{CELL_LINE.lower()}_heatmap.png'
if RUN_CDI and heatmap_path.exists():
    display(Markdown('## Heatmap'))
    display(Image(filename=str(heatmap_path), width=900))

overlay_paths = sorted(output_dir.glob('*/plate_overlay.png'))
if not overlay_paths:
    print('Plate overlay не найдены.')
else:
    for overlay_path in overlay_paths:
        display(Markdown(f'### {overlay_path.parent.name}'))
        display(Image(filename=str(overlay_path), width=900))


## 9. Скачать результаты ZIP-архивом

In [ ]:
import os
import shutil
from pathlib import Path
from google.colab import files

output_root = Path(os.environ['CLONO_OUTPUT'])
output_dir = output_root / OUTPUT_FOLDER
archive = shutil.make_archive(str(output_root / f'{OUTPUT_FOLDER}_results'), 'zip', root_dir=output_dir)
print('Архив готов:', archive)
files.download(archive)
